# KnowDecay — ML Model Training Notebook
**Trains:** Random Forest + XGBoost on Duolingo real data (or 500k synthetic fallback)

**Steps:**
1. Install dependencies
2. Load dataset (real or synthetic)
3. Train Random Forest + XGBoost
4. Compare accuracy
5. Download model files → put in `ml-service/models/saved/`

In [ ]:
# CELL 1 — Install dependencies
!pip install xgboost scikit-learn pandas numpy joblib -q
print('Done installing')

In [ ]:
# CELL 2 — Import libraries
import os, math
import numpy as np
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score
from xgboost import XGBRegressor, XGBClassifier

os.makedirs('models/saved', exist_ok=True)
print('Libraries ready')

In [ ]:
# CELL 3 — OPTION A: Upload Duolingo CSV from your computer
# Skip this cell if you want to use synthetic data (go to Cell 5)
from google.colab import files
uploaded = files.upload()  # select duolingo_data.csv or learning_traces.13m.csv
dataset_file = list(uploaded.keys())[0]
print(f'Uploaded: {dataset_file}')

In [ ]:
# CELL 4 — OPTION B: Download from Kaggle directly (need kaggle.json API key)
# Skip this cell if you already uploaded in Cell 3
# Get kaggle.json from: kaggle.com → Profile → Settings → API → Create New Token

from google.colab import files
print('Upload your kaggle.json file:')
files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d aravinii/duolingo-spaced-repetition-data -q
!unzip -q duolingo-spaced-repetition-data.zip
dataset_file = 'duolingo_data.csv'
print(f'Downloaded: {dataset_file}')

In [ ]:
# CELL 5 — Load dataset
# If dataset_file is defined above, use real data. Otherwise generate synthetic.

FEATURES = ['time_since_last_review', 'quiz_score', 'difficulty', 'review_count', 'study_duration']
MAX_ROWS  = 500_000  # change to None for full 13M (slower)

def load_duolingo(path, max_rows):
    print(f'Loading {path} (max {max_rows:,} rows)...')
    df_raw = pd.read_csv(path, nrows=max_rows)
    print(f'Columns: {df_raw.columns.tolist()}')
    seen_col    = 'history_seen'
    correct_col = 'history_correct'
    df = pd.DataFrame({
        'time_since_last_review': df_raw['delta'] / 3600.0,
        'quiz_score':             (df_raw[correct_col] / df_raw[seen_col].replace(0, 1) * 100).clip(0, 100),
        'difficulty':             1,
        'review_count':           df_raw[seen_col].clip(0, 20),
        'study_duration':         5,
        'retention':              df_raw['p_recall'].clip(0.0, 1.0),
    }).dropna()
    df = df[df['time_since_last_review'] > 0]
    return df

def generate_synthetic(n=500_000):
    print(f'Generating {n:,} synthetic rows...')
    rng = np.random.default_rng(42)
    DIFF_FACTOR = {0:1.5, 1:1.0, 2:0.7}
    t   = rng.exponential(72, n).clip(0.5, 720)
    s   = rng.beta(5, 2, n) * 100
    d   = rng.integers(0, 3, n)
    r   = rng.integers(0, 11, n)
    dur = rng.integers(10, 121, n)
    retention = np.array([
        max(0, min(1, math.exp(-ti / (24 * DIFF_FACTOR[di] * (1+si/100) * (1+ri*0.3) * (1+duri/200)))))
        for ti,si,ri,di,duri in zip(t,s,r,d,dur)
    ])
    retention = np.clip(retention + rng.normal(0, 0.025, n), 0, 1)
    return pd.DataFrame({'time_since_last_review':t,'quiz_score':s,'difficulty':d,'review_count':r,'study_duration':dur,'retention':retention})

# Auto-detect
try:
    df = load_duolingo(dataset_file, MAX_ROWS)
    source = 'duolingo_real'
except NameError:
    df = generate_synthetic(500_000)
    source = 'synthetic'

print(f'\nDataset: {len(df):,} rows [{source}]')
print(f'Retention range: {df.retention.min():.3f} — {df.retention.max():.3f}')
df.head()

In [ ]:
# CELL 6 — Prepare training/test split
X = df[FEATURES].values
y = df['retention'].values
y_binary = (y < 0.5).astype(int)  # 1 = will forget

X_train, X_test, y_train, y_test = train_test_split(X, y,        test_size=0.2, random_state=42)
_,       _,      yb_train,yb_test = train_test_split(X, y_binary, test_size=0.2, random_state=42)

print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')
print(f'Forget rate in test: {yb_test.mean():.1%}')

In [ ]:
# CELL 7 — Train Random Forest
print('Training Random Forest (200 trees)...')
rf = RandomForestRegressor(n_estimators=200, max_depth=10, min_samples_leaf=5, n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)

rf_preds = rf.predict(X_test)
rf_mae   = mean_absolute_error(y_test, rf_preds)
rf_r2    = r2_score(y_test, rf_preds)
print(f'Random Forest  →  MAE: {rf_mae:.4f}   R²: {rf_r2:.4f}')

In [ ]:
# CELL 8 — Train XGBoost Regressor
print('Training XGBoost Regressor (300 trees)...')
xgb_reg = XGBRegressor(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0,
    random_state=42, n_jobs=-1, verbosity=0
)
xgb_reg.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

xgb_preds = xgb_reg.predict(X_test)
xgb_mae   = mean_absolute_error(y_test, xgb_preds)
xgb_r2    = r2_score(y_test, xgb_preds)
print(f'XGBoost Regressor  →  MAE: {xgb_mae:.4f}   R²: {xgb_r2:.4f}')

winner = 'XGBoost' if xgb_mae < rf_mae else 'Random Forest'
print(f'\n Winner: {winner} (lower MAE wins)')

In [ ]:
# CELL 9 — Train XGBoost Classifier (will forget in 7 days?)
print('Training XGBoost Classifier...')
xgb_clf = XGBClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.1,
    subsample=0.8, random_state=42, n_jobs=-1,
    verbosity=0, eval_metric='logloss'
)
xgb_clf.fit(X_train, yb_train, verbose=False)
clf_acc = accuracy_score(yb_test, xgb_clf.predict(X_test))
print(f'XGBoost Classifier  →  Accuracy: {clf_acc:.4f}')

In [ ]:
# CELL 10 — Compare all models + feature importance
print('='*50)
print('MODEL COMPARISON')
print('='*50)
print(f'Random Forest   MAE: {rf_mae:.4f}   R²: {rf_r2:.4f}')
print(f'XGBoost Reg     MAE: {xgb_mae:.4f}   R²: {xgb_r2:.4f}')
print(f'XGBoost Clf     Accuracy: {clf_acc:.4f}')
print()

imp = pd.DataFrame({
    'Feature': FEATURES,
    'Random Forest': rf.feature_importances_,
    'XGBoost': xgb_reg.feature_importances_
}).sort_values('XGBoost', ascending=False)
print('FEATURE IMPORTANCE:')
print(imp.to_string(index=False))

In [ ]:
# CELL 11 — Save all models + metadata
best = 'xgboost' if xgb_mae < rf_mae else 'random_forest'

joblib.dump(rf,      'models/saved/random_forest.joblib')
joblib.dump(xgb_reg, 'models/saved/xgboost_regressor.joblib')
joblib.dump(xgb_clf, 'models/saved/xgboost_classifier.joblib')
joblib.dump({
    'best_model':  best,
    'dataset':     source,
    'n_rows':      len(df),
    'rf_mae':      rf_mae,
    'xgb_mae':     xgb_mae,
    'clf_acc':     clf_acc,
    'features':    FEATURES,
}, 'models/saved/meta.joblib')

print('All models saved!')
!ls -lh models/saved/

In [ ]:
# CELL 12 — Download all model files to your computer
# Then put them in: ml-service/models/saved/
from google.colab import files
files.download('models/saved/random_forest.joblib')
files.download('models/saved/xgboost_regressor.joblib')
files.download('models/saved/xgboost_classifier.joblib')
files.download('models/saved/meta.joblib')
print('4 files downloaded. Put them in: ml-service/models/saved/')